In [3]:
import os
import sys
import subprocess

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return os.path.isdir('/content')

def find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.isdir(os.path.join(path, 'sides_matching')):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            break
        path = parent
    for candidate in ['/content/sides-matching', '/content']:
        if os.path.isdir(os.path.join(candidate, 'sides_matching')):
            return candidate
    return None

def _purge_sides_matching_modules():
    for name in list(sys.modules):
        if name == 'sides_matching' or name.startswith('sides_matching.'):
            del sys.modules[name]

def ensure_repo_root():
    clone_dir = '/content/sides-matching'
    repo_url = 'https://github.com/abui-am/side-matching.git'
    root = find_repo_root()
    if root is None:
        if not in_colab():
            raise RuntimeError(
                'Could not find sides_matching. Open the sides-matching folder '
                'in Cursor and connect the Colab extension from this repo.'
            )
        print('Cloning side-matching repo...')
        subprocess.check_call(['git', 'clone', '--depth', '1', repo_url, clone_dir])
        root = clone_dir
    elif in_colab() and os.path.isdir(os.path.join(root, '.git')):
        print(f'Updating repo at {root} to origin/main...')
        subprocess.check_call(['git', '-C', root, 'fetch', '--depth', '1', 'origin', 'main'])
        subprocess.check_call(['git', '-C', root, 'reset', '--hard', 'origin/main'])
    _purge_sides_matching_modules()
    return root

def pip_install(*packages):
    cmd = [sys.executable, '-m', 'pip', 'install', *packages]
    print('>', ' '.join(cmd))
    subprocess.check_call(cmd)

repo_root = ensure_repo_root()

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

pip_install('wildlife-datasets', 'timm', 'scikit-image')
pip_install('git+https://github.com/WildlifeDatasets/wildlife-tools@main')
if not in_colab():
    pip_install('ipykernel')

DRIVE_DATA = '/content/drive/MyDrive/SeaTurtle'
DATASET_DIRS = ('AmvrakikosTurtles', 'ReunionTurtles', 'ZakynthosTurtles')

def has_datasets(path):
    return all(os.path.isdir(os.path.join(path, name)) for name in DATASET_DIRS)

def find_data_dir():
    if in_colab():
        from google.colab import drive
        drive.mount('/content/drive')
        if has_datasets(DRIVE_DATA):
            return DRIVE_DATA
        raise FileNotFoundError(
            f'Datasets not found at {DRIVE_DATA}. '
            'Put AmvrakikosTurtles, ReunionTurtles, ZakynthosTurtles in Drive Saya > SeaTurtle.'
        )
    local = os.path.join(repo_root, 'data')
    if has_datasets(local):
        return local
    raise FileNotFoundError(f'Datasets not found at {local}')

data_dir = find_data_dir()
print(f'Repo: {repo_root}')
print(f'Data: {data_dir} OK')

Mounted at /content/drive
Repo: /content/sides-matching
Data: /content/drive/MyDrive/SeaTurtle OK


In [4]:
import os
import sys
import numpy as np
import pandas as pd

if 'repo_root' not in globals():
    raise RuntimeError('Run the setup cell (cell 1) first.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from sides_matching import amvrakikos, reunion_green, reunion_hawksbill, zakynthos

root_data = data_dir
data = [
    ('Amvrakikos', os.path.join(root_data, 'AmvrakikosTurtles'), amvrakikos),
    ('ReunionGreen', os.path.join(root_data, 'ReunionTurtles'), reunion_green),
    ('ReunionHawksbill', os.path.join(root_data, 'ReunionTurtles'), reunion_hawksbill),
    ('Zakynthos', os.path.join(root_data, 'ZakynthosTurtles'), zakynthos),
]

In [5]:
def get_summary(df):
    spans = []
    for _, df_identity in df.groupby('identity'):
        span = df_identity['year'].max() - df_identity['year'].min()
        spans.append(span)

    data = {
        'Ind.': df['identity'].nunique(),
        'Photos': len(df),
        'Avg. span': np.mean(spans),
    }
    return data

In [6]:
summary = {}
for name, root, dataset_class in data:
    dataset = dataset_class(root)
    summary[name] = get_summary(dataset.df)

summary = pd.DataFrame(summary).T
summary[['Ind.', 'Photos']] = summary[['Ind.', 'Photos']].astype(np.int64)
summary    

,Ind.,Photos,Avg. span
Amvrakikos,50,200,4.440000
ReunionGreen,50,200,4.680000
ReunionHawksbill,34,136,3.411765
Zakynthos,40,160,2.525000
